In [2]:
import pandas as pd

## **Define Spam Checking Func using Rspamd**

https://github.com/rspamd/rspamd 
在docker里运行rspamd

In [30]:
# 确认容器是否可用
import subprocess

result = subprocess.run(
    ["docker", "ps"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

CONTAINER ID   IMAGE           COMMAND                CREATED       STATUS       PORTS                                                                     NAMES
a90165003daf   rspamd/rspamd   "/usr/bin/rspamd -f"   2 hours ago   Up 2 hours   0.0.0.0:11333-11334->11333-11334/tcp, [::]:11333-11334->11333-11334/tcp   rspamd




In [31]:
import subprocess
import json
from pathlib import Path

def check_spam(eml_path, container_name="rspamd"):
    eml_path = Path(eml_path)
    
    if not eml_path.exists():
        return {"ok": False, "error": f"File not found: {eml_path}"}
    
    cmd = f'docker exec -i {container_name} rspamc -j < "{eml_path}"'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode != 0:
        return {
            "ok": False,
            "error": result.stderr.strip(),
            "raw": result.stdout
        }
    
    try:
        data = json.loads(result.stdout)
        return {
            "ok": True,
            "score": data.get("score"),
            "required_score": data.get("required_score"),
            "action": data.get("action"),
            "symbols": data.get("symbols", {})
        }
    except Exception as e:
        return {
            "ok": False,
            "error": f"JSON parse error: {e}",
            "raw": result.stdout
        }

In [32]:
from pathlib import Path

eml_dir = Path("../data/raw/eml/")

files = list(eml_dir.rglob("*.eml"))

print("找到 eml 数量:", len(files))
print("示例:", files[:3])

找到 eml 数量: 228463
示例: [PosixPath('../data/raw/eml/2015/FW- FW- Promab Hybridoma Sequencing. - Laura Poling.eml'), PosixPath('../data/raw/eml/2015/Re- Custom Rabbit Monoclonal Antibody Service Inquiry[14].eml'), PosixPath('../data/raw/eml/2015/RE- Biocare-Weimin New Quotes[4].eml')]


In [33]:
# 测试一封邮件
sample_file = files[0]
print(sample_file)

res = check_spam(sample_file)
res

../data/raw/eml/2015/FW- FW- Promab Hybridoma Sequencing. - Laura Poling.eml


{'ok': True,
 'score': 11.6,
 'required_score': 15.0,
 'action': 'add header',
 'symbols': {'DATE_IN_PAST': {'name': 'DATE_IN_PAST',
   'score': 1.0,
   'metric_score': 1.0,
   'description': 'Message date is in the past',
   'options': ['95918']},
  'FROM_HAS_DN': {'name': 'FROM_HAS_DN',
   'score': 0.0,
   'metric_score': 0.0,
   'description': 'From header has a display name'},
  'FROM_EQ_ENVFROM': {'name': 'FROM_EQ_ENVFROM',
   'score': 0.0,
   'metric_score': 0.0,
   'description': 'From address is the same as the envelope'},
  'DMARC_POLICY_REJECT': {'name': 'DMARC_POLICY_REJECT',
   'score': 2.0,
   'metric_score': 2.0,
   'description': 'DMARC reject policy',
   'options': ['promab.com : No valid SPF, No valid DKIM', 'reject']},
  'TO_DN_ALL': {'name': 'TO_DN_ALL',
   'score': 0.0,
   'metric_score': 0.0,
   'description': 'All the recipients have display names'},
  'MIME_GOOD': {'name': 'MIME_GOOD',
   'score': -0.1,
   'metric_score': -0.1,
   'description': 'Known content-ty

## **Running Batch**

In [34]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import math

# =========================
# Config
# =========================
EML_DIR = Path("/Users/promab/anaconda_projects/email_agent/data/raw/eml")
OUT_DIR = Path("/Users/promab/anaconda_projects/email_agent/data/processed/spam_scan")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RSPAMC_CMD = ["docker", "exec", "-i", "rspamd", "rspamc", "-j"]  # if needed, you can add "-h", "localhost:11334"
MAX_WORKERS = 8
CHUNK_SIZE = 5000   # save every 5000 emails

In [38]:
import subprocess
import json
from pathlib import Path

# Example:
# RSPAMC_CMD = ["docker", "exec", "-i", "rspamd", "rspamc", "-j"]

def scan_one_eml(eml_path: Path):
    eml_path = Path(eml_path)

    base_result = {
        "file_path": str(eml_path),
        "file_name": eml_path.name,
        "ok": False,
        "error": None,
        "score": None,
        "required_score": None,
        "action": None,
        "is_spam": None,
        "symbols": None
    }

    try:
        if not eml_path.exists():
            base_result["error"] = f"file_not_found: {eml_path}"
            return base_result

        # Read raw eml bytes and pipe into rspamc stdin
        with open(eml_path, "rb") as f:
            eml_bytes = f.read()

        result = subprocess.run(
            RSPAMC_CMD,
            input=eml_bytes,
            capture_output=True,
            timeout=60
        )

        if result.returncode != 0:
            base_result["error"] = (
                result.stderr.decode("utf-8", errors="ignore").strip()
                if result.stderr
                else f"returncode={result.returncode}"
            )
            return base_result

        stdout = result.stdout.decode("utf-8", errors="ignore").strip()

        if not stdout:
            base_result["error"] = "empty_output"
            return base_result

        data = json.loads(stdout)

        score = data.get("score")
        required_score = data.get("required_score")
        action = data.get("action")
        symbols_raw = data.get("symbols", {})
        symbols = list(symbols_raw.keys()) if isinstance(symbols_raw, dict) else None

        # Spam rule: you can tighten or loosen later
        is_spam = (
        action in {"reject", "add header"}
        or (
            score is not None
            and required_score is not None
            and score >= required_score
        )
    )

        return {
            "file_path": str(eml_path),
            "file_name": eml_path.name,
            "ok": True,
            "error": None,
            "score": score,
            "required_score": required_score,
            "action": action,
            "is_spam": is_spam,
            "symbols": symbols
        }

    except subprocess.TimeoutExpired:
        base_result["error"] = "timeout"
        return base_result

    except json.JSONDecodeError as e:
        base_result["error"] = f"json_parse_error: {e}"
        return base_result

    except Exception as e:
        base_result["error"] = str(e)
        return base_result

In [50]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import math

# =========================
# Get all .eml files
# =========================
all_eml_files = sorted(EML_DIR.rglob("*.eml"))
print(f"Total .eml files found: {len(all_eml_files)}")

# =========================
# Batch scan + checkpoint save
# =========================
num_chunks = math.ceil(len(all_eml_files) / CHUNK_SIZE)
print(f"Total chunks: {num_chunks}")

for chunk_idx in range(num_chunks):
    out_file = OUT_DIR / f"spam_scan_chunk_{chunk_idx:03d}.parquet"

    # ✅ 如果已经存在 -> 跳过（断点续跑）
    if out_file.exists():
        print(f"Skipping chunk {chunk_idx} (already exists)")
        continue

    start = chunk_idx * CHUNK_SIZE
    end = min((chunk_idx + 1) * CHUNK_SIZE, len(all_eml_files))
    batch_files = all_eml_files[start:end]

    print(f"\nProcessing chunk {chunk_idx+1}/{num_chunks} | files {start} - {end-1}")

    rows = []
    buffer_size = 500   # ⭐ 每500条先落一次盘（防止丢）

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(scan_one_eml, fp): fp for fp in batch_files}

        for i, future in enumerate(as_completed(futures), 1):
            rows.append(future.result())

            # ✅ 每500条先写一次（临时文件）
            if i % buffer_size == 0:
                temp_df = pd.DataFrame(rows)
                temp_file = OUT_DIR / f"temp_chunk_{chunk_idx:03d}.parquet"
                temp_df.to_parquet(temp_file, index=False)
                print(f"  checkpoint saved ({i} rows)")

    # ✅ chunk完成后最终保存
    df_batch = pd.DataFrame(rows)
    df_batch.to_parquet(out_file, index=False)

    print(f"Saved final: {out_file}")

print("\nAll chunks finished.")

Total .eml files found: 228463
Total chunks: 46
Skipping chunk 0 (already exists)
Skipping chunk 1 (already exists)
Skipping chunk 2 (already exists)
Skipping chunk 3 (already exists)
Skipping chunk 4 (already exists)
Skipping chunk 5 (already exists)
Skipping chunk 6 (already exists)
Skipping chunk 7 (already exists)
Skipping chunk 8 (already exists)
Skipping chunk 9 (already exists)
Skipping chunk 10 (already exists)
Skipping chunk 11 (already exists)
Skipping chunk 12 (already exists)
Skipping chunk 13 (already exists)
Skipping chunk 14 (already exists)
Skipping chunk 15 (already exists)
Skipping chunk 16 (already exists)
Skipping chunk 17 (already exists)
Skipping chunk 18 (already exists)
Skipping chunk 19 (already exists)
Skipping chunk 20 (already exists)
Skipping chunk 21 (already exists)
Skipping chunk 22 (already exists)
Skipping chunk 23 (already exists)
Skipping chunk 24 (already exists)
Skipping chunk 25 (already exists)
Skipping chunk 26 (already exists)
Skipping chunk 27

In [3]:
df = pd.read_parquet(
    "/Users/promab/anaconda_projects/email_agent/data/processed/spam_scan/spam_scan_chunk_000.parquet"
)

df.head(50)

,file_path,file_name,ok,error,score,required_score,action,is_spam,symbols
0,/Users/promab/anaconda_projects/email_agent/da...,$35.7M Ferrari; Anne Hathaway's and Megan Fox'...,True,None,2.590000,15.0,no action,False,"[FROM_HAS_DN, FROM_NEQ_ENVFROM, REPLYTO_ADDR_E..."
1,/Users/promab/anaconda_projects/email_agent/da...,10 Best Practices in Vulnerability Management[...,True,None,9.990000,15.0,add header,True,"[DATE_IN_PAST, FROM_HAS_DN, FROM_NEQ_ENVFROM, ..."
2,/Users/promab/anaconda_projects/email_agent/da...,- Interchim - Happy New Year 2016 !.eml,True,None,5.400000,15.0,greylist,False,"[DATE_IN_PAST, FROM_HAS_DN, FROM_EQ_ENVFROM, H..."
3,/Users/promab/anaconda_projects/email_agent/da...,- Interchim - Happy New Year 2016 ! - Copy.eml,True,None,5.400000,15.0,greylist,False,"[DATE_IN_PAST, FROM_HAS_DN, FROM_EQ_ENVFROM, H..."
4,/Users/promab/anaconda_projects/email_agent/da...,1 Minute Survey – Win an Apple Watch - Copy.eml,True,None,3.278762,15.0,no action,False,"[DATE_IN_PAST, FROM_HAS_DN, FROM_NEQ_ENVFROM, ..."
5,/Users/promab/anaconda_projects/email_agent/da...,1 Minute Survey – Win an Apple Watch.eml,True,None,3.278762,15.0,no action,False,"[DATE_IN_PAST, FROM_HAS_DN, FROM_NEQ_ENVFROM, ..."
6,/Users/promab/anaconda_projects/email_agent/da...,10 Cost-saving Benefits of Vulnerability Manag...,True,None,9.990000,15.0,add header,True,"[DATE_IN_PAST, FROM_HAS_DN, FROM_NEQ_ENVFROM, ..."
7,/Users/promab/anaconda_projects/email_agent/da...,10 Cost-saving Benefits of Vulnerability Manag...,True,None,10.990000,15.0,add header,True,"[DATE_IN_PAST, FROM_HAS_DN, FROM_NEQ_ENVFROM, ..."
8,/Users/promab/anaconda_projects/email_agent/da...,10 Cost-saving Benefits of Vulnerability Manag...,True,None,9.990000,15.0,add header,True,"[DATE_IN_PAST, FROM_HAS_DN, FROM_NEQ_ENVFROM, ..."
9,/Users/promab/anaconda_projects/email_agent/da...,$35.7M Ferrari; Anne Hathaway's and Megan Fox'...,True,None,2.590000,15.0,no action,False,"[FROM_HAS_DN, FROM_NEQ_ENVFROM, REPLYTO_ADDR_E..."
